# Tutorial environment setup

Run this notebook once after cloning the tutorial on a new machine or with a new Julia depot. Run it again only when `Project.toml` or `Manifest.toml` changes, or when the local package installation has been cleared.

The numbered chapter notebooks activate this shared environment but do not reinstall it.


## 1. Activate the project and configure registries

Some SciBmad dependencies are distributed through the Bmad Julia registry. The checks below add General or the Bmad registry only when it is missing, so this cell is safe to run again.


In [ ]:
import Pkg
Pkg.activate(@__DIR__)

registry_names = Set(registry.name for registry in Pkg.Registry.reachable_registries())
"General" in registry_names || Pkg.Registry.add("General")

registries = Pkg.Registry.reachable_registries()
has_bmad_registry = any(registries) do registry
    repo = something(registry.repo, "")
    registry.name == "BmadRegistry" || occursin("bmad-sim/BmadRegistry.jl", repo)
end

if !has_bmad_registry
    Pkg.Registry.add(Pkg.RegistrySpec(url="https://github.com/bmad-sim/BmadRegistry.jl"))
end


## 2. Install the shared environment

This tutorial is tested against the official SciBmad 0.5.2 release, which adds the dynamic-aperture and phase-trombone APIs used in Chapter 12. Setup pins the `v0.5.2` release tag before `Pkg.instantiate()` installs the remaining packages and artifacts. To keep the initial setup focused, automatic precompilation of the full environment is disabled; only the core packages used throughout the tutorial are precompiled explicitly. Other packages will precompile when they are first loaded. Later chapter notebooks do not repeat this step.


In [ ]:
scibmad_release = "v0.5.2"
Pkg.add(
    Pkg.PackageSpec(
        url="https://github.com/bmad-sim/SciBmad.jl",
        rev=scibmad_release,
    );
    # Keep direct dependencies fixed while allowing SciBmad's required
    # transitive dependencies (for example NonlinearNormalForm) to update.
    preserve=Pkg.PRESERVE_DIRECT,
)
Pkg.instantiate(; allow_autoprecomp=false)

Pkg.precompile([
    "SciBmad",
    "Beamlines",
    "GTPSA",
    "CairoMakie",
])

println("Tutorial environment is ready. You can now open any numbered chapter notebook.")
